# 01 - Branch and X


## Metodo exacto

Branch and X pertenece a los algoritmos exactos de optimizacion. Su objetivo no
es solamente encontrar una solucion buena, sino encontrar el optimo global y
justificar que ninguna otra solucion factible es mejor.

Por eso no se considera una metaheuristica. Una metaheuristica acepta perder la
garantia de optimalidad para buscar soluciones de buena calidad en menos tiempo.
Branch and X mantiene una logica distinta: explora el espacio de busqueda con
una garantia matematica, aunque en algunos problemas eso pueda ser caro.

La idea no es revisar todas las soluciones una por una. La idea es organizar la
busqueda en subproblemas y descartar grupos completos de soluciones cuando se
puede demostrar que no pueden mejorar la mejor solucion conocida.


## Modelo de optimizacion

Un problema de optimizacion se construye con variables de decision, funcion
objetivo y restricciones.

Las variables de decision son lo que el algoritmo debe escoger. En un problema
binario pueden valer 0 o 1; en uno entero pueden tomar valores discretos; en uno
continuo pueden tomar valores reales.

La funcion objetivo mide la calidad de una solucion. Si el problema es de
minimizacion, valores menores son mejores. Si es de maximizacion, valores
mayores son mejores.

Las restricciones definen que soluciones son factibles. Una solucion con buen
valor objetivo no sirve si rompe alguna condicion del problema.

Branch and X trabaja especialmente bien cuando el problema puede separarse en
decisiones parciales. Cada decision parcial reduce el conjunto de soluciones que
todavia tiene sentido considerar.


## Arbol de busqueda

Branch and X representa la busqueda como un arbol.

La raiz contiene el problema completo. Cada nodo representa un subproblema, es
decir, una version del problema donde algunas decisiones ya fueron fijadas.

Al bajar por el arbol, el algoritmo agrega decisiones. Eso hace que cada nodo
represente un conjunto mas pequeno de soluciones posibles.

Esta estructura permite razonar por grupos: si un nodo no puede producir una
mejor solucion, entonces tampoco pueden hacerlo sus hijos. En ese caso se poda
toda la rama.


## Branching

Branching es la regla que divide un nodo en subproblemas hijos.

En variables binarias, una division comun es fijar una variable en 0 y en 1. En
variables enteras, una division comun es separar una variable segun el piso y el
techo de un valor fraccionario obtenido desde una relajacion.

Lo importante es que el branching no pierda soluciones factibles relevantes. Los
hijos deben cubrir las posibilidades del nodo padre, porque si una solucion
optima desaparece por la division, el metodo deja de ser exacto.

Una buena regla de branching puede reducir mucho el numero de nodos explorados.
No cambia el peor caso teorico, pero si puede cambiar fuertemente el rendimiento
practico.


## Bounding

Bounding es el calculo de una cota optimista para un subproblema.

En minimizacion, la cota es inferior: indica el mejor valor minimo que ese nodo
podria llegar a alcanzar. En maximizacion, la cota es superior: indica el mejor
valor maximo que ese nodo podria alcanzar.

La cota suele venir de una relajacion. Una relajacion es una version mas facil
del problema, por ejemplo permitir valores fraccionarios donde antes solo habia
variables enteras.

La relajacion puede entregar una solucion que no sea factible para el problema
original, pero sirve para estimar que tan prometedor es un nodo.

Mientras mas ajustada sea la cota, mas facil es podar. Una cota muy debil obliga
al algoritmo a explorar mas nodos.


## Incumbente

El incumbente es la mejor solucion factible encontrada hasta el momento.

En minimizacion, es la solucion con menor valor conocida. En maximizacion, es la
solucion con mayor valor conocida.

El incumbente es importante porque transforma una solucion encontrada en un
criterio de descarte. Si la cota de un nodo muestra que ese subproblema no puede
mejorar al incumbente, entonces no vale la pena seguir explorandolo.

Por eso encontrar un buen incumbente temprano ayuda mucho: mientras mejor sea la
referencia, mas ramas se pueden podar.


## Pruning

Pruning significa podar un nodo del arbol.

Un nodo se puede podar si es infactible, si su cota no puede mejorar al
incumbente o si ya representa una solucion completa que no necesita seguir
dividiendose.

La poda es la diferencia entre una busqueda inteligente y una enumeracion bruta.
Branch and X sigue pudiendo ser exponencial en el peor caso, pero en la practica
su rendimiento depende de cuantas ramas logra descartar antes de abrirlas.


## Estructura en Python

La forma tecnica de entender Branch and Bound es verlo como una funcion general
que no conoce el problema especifico. El algoritmo recibe funciones externas que
definen como se comporta el problema.

En el notebook, esa estructura se implementa con estas piezas:

- `root`: nodo inicial.
- `branch(node)`: genera los hijos de un nodo.
- `bound(node)`: calcula la cota optimista del nodo.
- `is_feasible(node)`: indica si el nodo todavia puede contener soluciones validas.
- `is_complete(node)`: indica si el nodo ya representa una solucion completa.
- `objective(node)`: calcula el valor real de una solucion completa.
- `incumbent`: mejor solucion factible encontrada hasta ahora.

Esta separacion es importante: Branch and Bound no cambia de idea cuando cambia
el problema. Lo que cambia son las funciones que definen la representacion, la
cota y la forma de ramificar.


In [1]:
from dataclasses import dataclass
from heapq import heappush, heappop
from itertools import count
from math import inf

@dataclass
class BranchAndBoundResult:
    best_solution: object
    best_value: float | None
    explored: int
    pruned_by_bound: int
    pruned_infeasible: int


def branch_and_bound(
    root,
    branch,
    bound,
    is_feasible,
    is_complete,
    objective,
    *,
    sense="min",
    initial_solution=None,
):
    """
    Branch and Bound generico.

    El algoritmo no sabe si el problema es TSP, mochila, scheduling u otro.
    Esa informacion entra por las funciones branch, bound, is_feasible,
    is_complete y objective.
    """
    if sense not in {"min", "max"}:
        raise ValueError("sense debe ser 'min' o 'max'")

    sign = 1 if sense == "min" else -1

    best_solution = initial_solution
    best_value = objective(initial_solution) if initial_solution is not None else None
    best_key = sign * best_value if best_value is not None else inf

    pending = []
    node_id = count()
    heappush(pending, (sign * bound(root), next(node_id), root))

    explored = 0
    pruned_by_bound = 0
    pruned_infeasible = 0

    while pending:
        _, _, node = heappop(pending)
        explored += 1

        if not is_feasible(node):
            pruned_infeasible += 1
            continue

        node_key = sign * bound(node)
        if node_key >= best_key:
            pruned_by_bound += 1
            continue

        if is_complete(node):
            value = objective(node)
            value_key = sign * value
            if value_key < best_key:
                best_solution = node
                best_value = value
                best_key = value_key
            continue

        for child in branch(node):
            if not is_feasible(child):
                pruned_infeasible += 1
                continue

            child_key = sign * bound(child)
            if child_key >= best_key:
                pruned_by_bound += 1
                continue

            heappush(pending, (child_key, next(node_id), child))

    return BranchAndBoundResult(
        best_solution=best_solution,
        best_value=best_value,
        explored=explored,
        pruned_by_bound=pruned_by_bound,
        pruned_infeasible=pruned_infeasible,
    )

## Lectura del codigo

El codigo implementa una estructura general de Branch and Bound.

La clase `BranchAndBoundResult` solo ordena la salida del algoritmo: guarda la
mejor solucion, su valor, cuantos nodos se exploraron y cuantas podas ocurrieron.

La funcion `branch_and_bound` recibe un nodo raiz y varias funciones del
problema. Esto permite que el mismo algoritmo sirva para distintos modelos. Si
el problema cambia, no se reescribe Branch and Bound completo; se cambian las
funciones que describen el problema.

La variable `best_solution` representa el incumbente. `best_value` guarda su
valor objetivo. Si el problema es de minimizacion, el algoritmo busca disminuir
ese valor; si es de maximizacion, usa un cambio de signo para mantener la misma
logica interna.

La lista `pending` guarda los nodos que todavia pueden explorarse. Se maneja con
una cola de prioridad, de modo que el algoritmo puede revisar primero los nodos
mas prometedores segun su cota.

En cada iteracion se extrae un nodo, se revisa si es factible, se compara su cota
con el incumbente y se decide si conviene podarlo. Si el nodo ya es una solucion
completa, se evalua con `objective`. Si mejora al incumbente, se actualiza la
mejor solucion.

Si el nodo no se poda y todavia no es completo, se generan sus hijos con
`branch(node)`. Cada hijo vuelve a pasar por la misma logica: factibilidad, cota,
poda o exploracion.

La idea importante es que el codigo no enumera soluciones a ciegas. Cada nodo se
abre solo si todavia tiene posibilidad de mejorar lo mejor encontrado hasta el
momento.


## Ejemplo 1 - TSP

En este ejemplo se usa Branch and Bound sobre un TSP pequeno. La solucion es una ruta completa, pero cada nodo del arbol representa una ruta parcial.

El branching agrega una ciudad no visitada. La cota estima de forma optimista cuanto podria costar terminar la ruta. Si esa cota no puede mejorar al incumbente, se poda la rama.


In [2]:
import math

coords = [
    (0.10, 0.20), (0.25, 0.85), (0.50, 0.55),
    (0.80, 0.75), (0.90, 0.15), (0.45, 0.10),
    (0.15, 0.55),
]

n = len(coords)

def distance(a, b):
    ax, ay = coords[a]
    bx, by = coords[b]
    return math.hypot(ax - bx, ay - by)

D = [[distance(i, j) for j in range(n)] for i in range(n)]

def nearest_neighbor_tour():
    tour = [0]
    remaining = set(range(1, n))
    while remaining:
        last = tour[-1]
        nxt = min(remaining, key=lambda city: D[last][city])
        tour.append(nxt)
        remaining.remove(nxt)
    return (tuple(tour), frozenset(), tour_length(tour))

def tour_length(tour):
    return sum(D[tour[i]][tour[(i + 1) % len(tour)]] for i in range(len(tour)))

root = ((0,), frozenset(range(1, n)), 0.0)

def branch_tsp(node):
    tour, remaining, cost_so_far = node
    last = tour[-1]
    for city in sorted(remaining):
        yield (
            tour + (city,),
            remaining - {city},
            cost_so_far + D[last][city],
        )

def bound_tsp(node):
    tour, remaining, cost_so_far = node
    if not remaining:
        return cost_so_far + D[tour[-1]][0]

    candidates = list(remaining) + [0]
    optimistic = cost_so_far
    for city in [tour[-1]] + list(remaining):
        optimistic += min(D[city][other] for other in candidates if other != city)
    return optimistic

def is_feasible_tsp(node):
    return True

def is_complete_tsp(node):
    return len(node[1]) == 0

def objective_tsp(node):
    tour, _, cost_so_far = node
    return cost_so_far + D[tour[-1]][0]

result = branch_and_bound(
    root,
    branch_tsp,
    bound_tsp,
    is_feasible_tsp,
    is_complete_tsp,
    objective_tsp,
    sense="min",
    initial_solution=nearest_neighbor_tour(),
)

print("Mejor tour:", result.best_solution[0])
print("Distancia:", round(result.best_value, 4))
print("Nodos explorados:", result.explored)
print("Podas por cota:", result.pruned_by_bound)
print("Podas por infactibilidad:", result.pruned_infeasible)

Mejor tour: (0, 6, 1, 2, 3, 4, 5)
Distancia: 2.8459
Nodos explorados: 35
Podas por cota: 25
Podas por infactibilidad: 0


## Lectura del ejemplo TSP

El objetivo es minimizar la distancia total del tour. El nodo guarda tres datos: la ruta parcial, las ciudades pendientes y el costo acumulado.

La funcion `branch_tsp` genera hijos agregando una ciudad pendiente. La funcion `bound_tsp` calcula una cota inferior: toma el costo actual y suma conexiones baratas necesarias para completar la ruta.

Este ejemplo muestra por que Branch and Bound no enumera rutas completas a ciegas. Muchas rutas parciales se descartan antes de completarse.



La comparacion con fuerza bruta ayuda a entender la gracia del metodo.

Fuerza bruta construiria todos los tours posibles y recien al final compararia.
Branch and Bound, en cambio, trabaja con rutas parciales. Si una ruta parcial ya
tiene una cota mala, no espera a completarla: poda esa rama.

Por eso el nodo no es necesariamente una solucion final. En TSP puede ser algo
como `0 -> 6 -> 1`, que todavia no visita todas las ciudades. Ese nodo representa
muchas rutas futuras posibles. La cota responde una pregunta clave: incluso en
el mejor caso, podria esta ruta parcial superar al incumbente?


## Ejemplo 2 - Mochila 0/1

La mochila 0/1 es un caso muy natural para Branch and Bound porque cada decision es binaria: tomar o no tomar un item.

El branching fija la decision del siguiente item. La cota se obtiene con una relajacion fraccional: se permite llenar la mochila con fracciones para estimar un valor maximo optimista.


In [3]:
items = [
    {"name": "A", "value": 20, "weight": 2},
    {"name": "B", "value": 30, "weight": 5},
    {"name": "C", "value": 35, "weight": 7},
    {"name": "D", "value": 12, "weight": 3},
    {"name": "E", "value": 3,  "weight": 1},
    {"name": "F", "value": 50, "weight": 9},
]
capacity = 15
items = sorted(items, key=lambda item: item["value"] / item["weight"], reverse=True)

root = (0, tuple(), 0, 0)  # index, elegidos, peso, valor

def branch_knapsack(node):
    index, chosen, weight, value = node
    if index >= len(items):
        return []
    item = items[index]
    skip = (index + 1, chosen, weight, value)
    take = (
        index + 1,
        chosen + (item["name"],),
        weight + item["weight"],
        value + item["value"],
    )
    return [take, skip]

def bound_knapsack(node):
    index, chosen, weight, value = node
    if weight > capacity:
        return -math.inf

    remaining_capacity = capacity - weight
    optimistic = value
    for item in items[index:]:
        if item["weight"] <= remaining_capacity:
            optimistic += item["value"]
            remaining_capacity -= item["weight"]
        else:
            optimistic += item["value"] * (remaining_capacity / item["weight"])
            break
    return optimistic

def is_feasible_knapsack(node):
    return node[2] <= capacity

def is_complete_knapsack(node):
    return node[0] == len(items)

def objective_knapsack(node):
    return node[3]

result = branch_and_bound(
    root,
    branch_knapsack,
    bound_knapsack,
    is_feasible_knapsack,
    is_complete_knapsack,
    objective_knapsack,
    sense="max",
)

print("Items elegidos:", result.best_solution[1])
print("Valor total:", result.best_value)
print("Peso total:", result.best_solution[2])
print("Nodos explorados:", result.explored)
print("Podas por cota:", result.pruned_by_bound)
print("Podas por infactibilidad:", result.pruned_infeasible)

Items elegidos: ('A', 'B', 'C', 'E')
Valor total: 88
Peso total: 15
Nodos explorados: 14
Podas por cota: 5
Podas por infactibilidad: 3


## Lectura del ejemplo Mochila

En este ejemplo el problema es de maximizacion. Por eso la cota es superior: indica el mejor valor que podria alcanzarse desde un nodo.

Si la cota superior de un nodo no supera al incumbente, no vale la pena seguir ramificando. Aunque la relajacion fraccional no sea una solucion valida de la mochila 0/1, sirve como limite optimista para podar.


## Branch and X

La letra X indica que la estructura base de ramificar y podar puede combinarse
con distintas tecnicas.

Branch and Bound usa cotas para decidir que ramas descartar.

Branch and Cut agrega cortes, que son restricciones validas para fortalecer la
relajacion. Un corte elimina soluciones artificiales de la relajacion, pero no
elimina soluciones factibles del problema original.



En TSP, por ejemplo, una relajacion puede permitir dos ciclos separados. Eso
puede verse barato matematicamente, pero no es un tour valido porque el viajante
debe hacer un solo ciclo que pase por todas las ciudades.

Un corte agrega una restriccion que obliga a conectar esos grupos. No esta
"inventando" una solucion; esta eliminando una respuesta falsa que aparecio
porque el modelo relajado era demasiado permisivo.

Branch and Price genera variables o columnas durante la busqueda. Esto se usa
cuando el modelo completo tendria demasiadas variables para construirlas todas
desde el inicio.

Las variantes cambian la herramienta matematica, pero mantienen la misma idea
base: dividir el problema y usar informacion optimista para evitar explorar
partes inutiles del arbol.


## Complejidad

Branch and X puede tener complejidad exponencial en el peor caso. Si las cotas
son debiles o el branching es malo, el arbol puede crecer demasiado y acercarse a
una enumeracion completa.

En la practica, el rendimiento depende de cuatro factores: que tan buena es la
cota, que tan rapido aparece un buen incumbente, que variable se elige para
ramificar y en que orden se exploran los nodos.

Por eso no basta decir que el metodo es exacto. Tambien hay que entender que la
eficiencia depende de la calidad de la informacion usada para podar.


## Cuando se usa

Branch and X se usa cuando la garantia de optimalidad es importante o cuando el
problema tiene una estructura que permite buenas cotas.

Es comun en optimizacion combinatoria y programacion entera: asignacion,
scheduling, rutas, seleccion de proyectos, diseno de redes y problemas con
decisiones discretas.

Cuando el problema es muy grande y no se necesita una prueba de optimalidad, una
heuristica o metaheuristica puede ser mas conveniente. Por eso este metodo sirve
como contraste: muestra lo que se gana y lo que se paga al exigir garantia.


## Resumen

Branch and X es una familia de algoritmos exactos. Su objetivo es encontrar el
optimo global y demostrar que lo encontro.

Su estructura se entiende con tres acciones: ramificar el problema, calcular
cotas y podar ramas que no pueden mejorar al incumbente.

No es una metaheuristica porque no se conforma con una solucion buena sin
prueba. Su fortaleza es la garantia; su debilidad es que puede volverse costoso
cuando el arbol de busqueda crece demasiado.
